<a href="https://colab.research.google.com/github/marcocslima/rag_tests/blob/main/RAG_testes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#RAG MULTIMODAL

##Setup

In [ ]:
%pip install -q -U google-genai
%pip install -q -U llama-index
%pip install -q -U llama-index-llms-google-genai
%pip install -q -U llama-index-embeddings-google-genai
%pip install -q -U llama-index-readers-web
%pip install -q -U llama-index-vector-stores-chroma
%pip install -q -U chromadb
%pip install -q -U bs4
%pip install -q -U llama-index-embeddings-huggingface sentence-transformers

In [ ]:
!pip install pymupdf pytesseract
!apt-get install -y tesseract-ocr tesseract-ocr-por

In [ ]:
import os
from google.colab import userdata

GOOGLE_API_KEY = userdata.get('GEMINI_API_KEY')
os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY

In [ ]:
from bs4 import BeautifulSoup
from IPython.display import Markdown, display

from llama_index.core import Document, Settings, StorageContext, VectorStoreIndex, PromptTemplate
from llama_index.readers.web import SimpleWebPageReader
from llama_index.embeddings.google_genai import GoogleGenAIEmbedding
from llama_index.llms.google_genai import GoogleGenAI
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

import chromadb

import nest_asyncio
nest_asyncio.apply()

In [ ]:
# === CARREGUE SEUS DADOS AQUI ===
# Exemplo com páginas web — troque pelas URLs que quiser
# urls = [
#     "https://blog.google/technology/google-deepmind/gemini-model-thinking-updates-march-2025/",
#     # Adicione mais URLs aqui
# ]

# web_documents = SimpleWebPageReader().load_data(urls)

# Extrair texto limpo do HTML
all_documents = []
# for doc in web_documents:
#     soup = BeautifulSoup(doc.text, 'html.parser')
#     paragraphs = soup.find_all('p')
#     text = "\n".join(p.text for p in paragraphs)
#     if text.strip():
#         all_documents.append(Document(text=text, metadata=doc.metadata))

# print(f"✅ {len(all_documents)} documento(s) carregado(s)")

In [ ]:
import fitz  # PyMuPDF
import pytesseract
from PIL import Image
import io
import glob
from llama_index.core import Document

def extract_pdf_text(pdf_path):
    doc = fitz.open(pdf_path)
    full_text = ""
    for page_num, page in enumerate(doc):
        text = page.get_text()
        if len(text.strip()) < 50:  # página é imagem → OCR
            pix = page.get_pixmap(dpi=300)
            img = Image.open(io.BytesIO(pix.tobytes("png")))
            text = pytesseract.image_to_string(img, lang="por")
            print(f"  🔍 OCR na página {page_num + 1}")
        full_text += text + "\n"
    doc.close()
    return full_text

# Ajuste o caminho para onde estão seus PDFs
pdf_files = glob.glob("/content/drive/MyDrive/Tmp/RAG_DOCS/*.pdf")

all_documents = []
for pdf_path in pdf_files:
    print(f"📄 Processando: {pdf_path}")
    text = extract_pdf_text(pdf_path)
    all_documents.append(Document(text=text, metadata={"source": pdf_path}))

print(f"\n✅ {len(all_documents)} documentos carregados")

# Verificar se agora está legível
print("\n--- Amostra do texto extraído ---")
print(all_documents[0].text[:500])

In [ ]:
# # Se quiser carregar arquivos do seu Google Drive ou uploads:
# # from google.colab import drive
# # drive.mount('/content/drive')

# from llama_index.core import SimpleDirectoryReader

# # Faça upload de arquivos para /content/docs/ ou aponte para o Drive
# # !mkdir -p /content/docs
# # Exemplo: copie PDFs para /content/docs/

# local_docs = SimpleDirectoryReader("/content/drive/MyDrive/Tmp/RAG_DOCS").load_data()
# all_documents.extend(local_docs)
# print(f"✅ Total: {len(all_documents)} documento(s)")

In [ ]:
# Modelo de embedding aberto Qwen/Qwen3-Embedding-0.6B
# embed_model = HuggingFaceEmbedding(
#     model_name="Qwen/Qwen3-Embedding-0.6B",  # ou "Qwen/Qwen3-Embedding-8B"
# )

# Modelo de embedding aberto BAAI/bge-m3
embed_model = HuggingFaceEmbedding(
    model_name="BAAI/bge-m3",
    query_instruction="Represent this sentence for searching relevant passages: ",
)

# Modelo de embedding para texto
# embed_model = GoogleGenAIEmbedding(
#     model_name="models/gemini-embedding-001",
#     task_type="RETRIEVAL_DOCUMENT",
# )

# Modelo de embedding multimodal (texto, imagem, PDF, áudio, vídeo)
# embed_model = GoogleGenAIEmbedding(
#     model_name="models/gemini-embedding-2-preview",
#     task_type="RETRIEVAL_DOCUMENT",  # otimizado para RAG
# )

# LLM para geração de respostas
llm = GoogleGenAI(model_name="models/gemini-2.5-flash")

# Configurações globais do LlamaIndex
Settings.llm = llm
Settings.embed_model = embed_model

# ChromaDB persistente em disco
chroma_client = chromadb.PersistentClient(path="/content/drive/MyDrive/Tmp/RAG_DBS/CHROMA/chroma_db")

#x-x-x-x-x-x-x-x-x-x-x-x-x-x-x-x-x-x-x-x-x-x-x-x-x-x-x-x-x-x-x-x-x-x-x-x-x-x
# Se precisar deletar a collection (CUIDADO!!!)
#chroma_client.delete_collection("rag_collection")
#x-x-x-x-x-x-x-x-x-x-x-x-x-x-x-x-x-x-x-x-x-x-x-x-x-x-x-x-x-x-x-x-x-x-x-x-x-x

# Criar ou reconectar a collection
chroma_collection = chroma_client.get_or_create_collection("rag_collection")

# Vector store + storage context
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)
storage_context = StorageContext.from_defaults(vector_store=vector_store)

print("✅ Modelos e ChromaDB configurados")

In [25]:
from llama_index.core import Settings

print(f"Embed model: {Settings.embed_model}")
print(f"Model name: {Settings.embed_model.model_name}")

Embed model: model_name='BAAI/bge-m3' embed_batch_size=10 callback_manager=<llama_index.core.callbacks.base.CallbackManager object at 0x7f9ebd72f9e0> num_workers=None embeddings_cache=None rate_limiter=None max_length=8192 normalize=True query_instruction='Represent this sentence for searching relevant passages: ' text_instruction=None cache_folder=None show_progress_bar=False
Model name: BAAI/bge-m3


##Run

In [ ]:
# Ajuste de acordo com as restrições das APIs:
# Settings.chunk_size = 512       # chunks menores = menos tokens por request
# Settings.chunk_overlap = 50

Settings.chunk_size = 1024      # chunks maiores = mais contexto por trecho
Settings.chunk_overlap = 128    # overlap maior = menos perda de info entre chunks

In [ ]:
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.callbacks import CallbackManager, TokenCountingHandler
import time

# Chunks maiores = melhor qualidade (sem limite de API)
parser = SentenceSplitter(chunk_size=1024, chunk_overlap=128)
nodes = parser.get_nodes_from_documents(all_documents)

total = len(nodes)
print(f"📄 Total de chunks: {total}\n")

# Indexar com progresso
BATCH_SIZE = 10
start = time.time()

for i in range(0, total, BATCH_SIZE):
    batch = nodes[i:i + BATCH_SIZE]
    if i == 0:
        index = VectorStoreIndex(batch, storage_context=storage_context)
    else:
        index.insert_nodes(batch)

    done = min(i + BATCH_SIZE, total)
    pct = (done / total) * 100
    elapsed = time.time() - start
    eta = (elapsed / done) * (total - done) if done > 0 else 0
    print(f"  ⏳ {done}/{total} chunks ({pct:.1f}%) — tempo restante: ~{eta:.0f}s")

elapsed_total = time.time() - start
print(f"\n✅ Índice criado com {total} chunks no ChromaDB em {elapsed_total:.1f}s")

In [ ]:
template = """Você é um assistente especializado em responder perguntas.
Use o contexto abaixo para responder à pergunta do usuário.
Se não souber a resposta, diga que não sabe.
Responda de forma concisa em no máximo 5 frases, em português.

Pergunta: {query_str}
Contexto: {context_str}
Resposta:"""

qa_prompt = PromptTemplate(template)

query_engine = index.as_query_engine(
    text_qa_template=qa_prompt,
    similarity_top_k=10,  # número de chunks relevantes retornados
)

print("✅ Query engine pronto!")

In [ ]:
pergunta = "Qual o texto do artigo 166 da Lei Complementar 460/2008"

response = query_engine.query(pergunta)
display(Markdown(response.response))

#RAG MODELO LOCAL

In [ ]:
!pip install -q sentence-transformers chromadb pymupdf

In [ ]:
import fitz  # PyMuPDF
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.utils import embedding_functions

# 1. Configurar o banco vetorial (salvo no drive para não perder)
# DICA: Substitua o caminho abaixo por um caminho no seu Drive montado
client = chromadb.PersistentClient(path="./base_vetorial_lei")

# 2. Carregar o modelo de embedding
model_name = 'BAAI/bge-m3'
embedding_func = embedding_functions.SentenceTransformerEmbeddingFunction(model_name=model_name)

# 3. Criar a coleção no banco
collection = client.get_or_create_collection(name="minha_lei", embedding_function=embedding_func)

# 4. Função para ler PDF e dividir em pedaços (chunks)
def processar_pdf(caminho_pdf):
    doc = fitz.open(caminho_pdf)
    textos = []
    ids = []

    # Divide por páginas ou por parágrafos. Aqui, dividiremos por páginas para simplificar
    for i, pagina in enumerate(doc):
        texto = pagina.get_text()
        if len(texto) > 100:  # Ignora páginas muito vazias
            textos.append(texto)
            ids.append(f"pag_{i}")

    return textos, ids

# 5. Adicionar ao ChromaDB
# Coloque o caminho do seu PDF aqui
caminho_do_seu_pdf = "/content/drive/MyDrive/Tmp/lei_complementar-460-2008.pdf"
textos, ids = processar_pdf(caminho_do_seu_pdf)

collection.add(
    documents=textos,
    ids=ids
)

print(f"Adicionados {len(textos)} trechos da lei ao banco!")

In [ ]:
query = "qual o texto do artigo 166 da LC 460?"

results = collection.query(
    query_texts=[query],
    n_results=5 # Pega os 2 trechos mais relevantes
)

print("--- Trechos mais relevantes encontrados ---")
for doc in results['documents'][0]:
    print(doc)
    print("-" * 20)